In [1]:
import pandas as pd

df = pd.read_csv('data review gofood.csv')

# Ambil 1 baris per Nomor Pesanan (deduplikasi)
orders = df.drop_duplicates(subset='Nomor Pesanan').copy()

# Pilih kolom yang relevan untuk level order
orders = orders[[
    'Nomor Pesanan',
    'Tanggal Transaksi',
    'Rating untuk Resto',
    'Review',
    'Tag yang Dipilih'
]].reset_index(drop=True)

# Rename kolom ke format SQL-friendly
orders.columns = [
    'order_id',
    'order_datetime',
    'rating',
    'review',
    'tags_raw'
]

# Parse datetime
orders['order_datetime'] = pd.to_datetime(orders['order_datetime'])

# Tambah kolom turunan yang berguna untuk SQL nanti
orders['order_hour'] = orders['order_datetime'].dt.hour
orders['order_date'] = orders['order_datetime'].dt.date
orders['has_review'] = orders['review'].notna().astype(int)
orders['has_tags'] = orders['tags_raw'].notna().astype(int)

print(f"Total orders: {len(orders)}")
print(orders.head())

Total orders: 90
       order_id order_datetime  rating  \
0  F-2616596270     2024-10-02       4   
1  F-2622650317     2024-10-07       5   
2  F-2625760663     2024-10-10       5   
3  F-2626691980     2024-10-11       5   
4  F-2629362503     2024-10-13       3   

                                              review tags_raw  order_hour  \
0                                                NaN      NaN           0   
1                                                NaN      NaN           0   
2                                                NaN      NaN           0   
3                                                NaN      NaN           0   
4  kali ini saya berikan bintang 3 saja\nsebelum2...      NaN           0   

   order_date  has_review  has_tags  
0  2024-10-02           0         0  
1  2024-10-07           0         0  
2  2024-10-10           0         0  
3  2024-10-11           0         0  
4  2024-10-13           1         0  


In [2]:

items_raw = df[['Nomor Pesanan', 'Nama Menu']].drop_duplicates().copy()
# drop_duplicates di sini menghapus baris yang Nomor Pesanan DAN Nama Menu-nya sama persis
# Hasilnya: 1 baris per order, baru kemudian di-explode per menu

items_raw['menu_list'] = items_raw['Nama Menu'].str.split(',')
items_exploded = items_raw.explode('menu_list')

items_exploded['menu_name'] = items_exploded['menu_list'].str.strip()

order_items = items_exploded[['Nomor Pesanan', 'menu_name']].copy()
order_items.columns = ['order_id', 'menu_name']
order_items = order_items.reset_index(drop=True)
order_items.insert(0, 'item_id', order_items.index + 1)

print(f"Total order items: {len(order_items)}")

Total order items: 149


In [3]:
# Ambil order yang punya tags
tags_raw = orders[orders['tags_raw'].notna()][['order_id', 'tags_raw']].copy()

# Split dan explode
tags_raw['tag_list'] = tags_raw['tags_raw'].str.split(',')
tags_exploded = tags_raw.explode('tag_list')

# Bersihkan
tags_exploded['tag_name'] = tags_exploded['tag_list'].str.strip()

# Sederhanakan nama tag: buang prefix "CANNED_RESPONSE_"
tags_exploded['tag_name'] = tags_exploded['tag_name'].str.replace('CANNED_RESPONSE_', '', regex=False)

# Susun tabel final
order_tags = tags_exploded[['order_id', 'tag_name']].reset_index(drop=True)
order_tags.insert(0, 'tag_id', order_tags.index + 1)

print(f"Total tags: {len(order_tags)}")
print(order_tags.head(10))

Total tags: 137
   tag_id      order_id   tag_name
0       1  F-2631519399      TASTE
1       2  F-2633790827    PORTION
2       3  F-2633790827  FRESHNESS
3       4  F-2633790827      TASTE
4       5  F-2634627942    PORTION
5       6  F-2634627942      VALUE
6       7  F-2643125700  FRESHNESS
7       8  F-2643125700    HYGIENE
8       9  F-2643125700      TASTE
9      10  F-2643125700      PRICE


In [4]:
# Pastikan format datetime menyertakan waktu
orders['order_datetime'] = pd.to_datetime(orders['order_datetime'])

# Saat export ke CSV, jangan hilangkan komponen waktu
orders.to_csv('orders.csv', index=False)

In [5]:
orders.to_csv('orders.csv', index=False)
order_items.to_csv('order_items.csv', index=False)
order_tags.to_csv('order_tags.csv', index=False)

print("=== Export selesai ===")
print(f"orders.csv        → {len(orders)} baris")
print(f"order_items.csv   → {len(order_items)} baris")
print(f"order_tags.csv    → {len(order_tags)} baris")

=== Export selesai ===
orders.csv        → 90 baris
order_items.csv   → 149 baris
order_tags.csv    → 137 baris


In [6]:
import sqlite3
conn = sqlite3.connect('db_order.db')

# Simpan datetime sebagai string lengkap dengan waktu
orders['order_datetime'] = orders['order_datetime'].dt.strftime('%Y-%m-%d %H:%M:%S')
orders.to_sql('orders', conn, if_exists='replace', index=False)

90

In [2]:
import pandas as pd
import sqlite3

# Baca dari xlsx langsung, bukan csv
df = pd.read_excel('data review gofood.xlsx')

# Cek hasilnya
print(df['Tanggal Transaksi'].head(5))
print(df['Tanggal Transaksi'].dtype)

0   2024-10-02 18:55:00
1   2024-10-02 18:55:00
2   2024-10-02 18:55:00
3   2024-10-07 22:18:00
4   2024-10-07 22:18:00
Name: Tanggal Transaksi, dtype: datetime64[ns]
datetime64[ns]


In [4]:
import pandas as pd
import sqlite3

# Baca dari xlsx
df = pd.read_excel('data review gofood.xlsx')

# Deduplikasi per order
orders = df.drop_duplicates(subset='Nomor Pesanan').copy()
orders = orders[[
    'Nomor Pesanan', 'Tanggal Transaksi', 'Rating untuk Resto', 'Review', 'Tag yang Dipilih'
]]
orders.columns = ['order_id', 'order_datetime', 'rating', 'review', 'tags_raw']

# Simpan komponen waktu secara eksplisit
orders['order_datetime'] = pd.to_datetime(orders['order_datetime']).dt.strftime('%Y-%m-%d %H:%M:%S')
orders['order_date'] = pd.to_datetime(orders['order_datetime']).dt.strftime('%Y-%m-%d')
orders['order_hour'] = pd.to_datetime(orders['order_datetime']).dt.hour
orders['has_review'] = orders['review'].notna().astype(int)
orders['has_tags'] = orders['tags_raw'].notna().astype(int)

# Verifikasi sebelum import
print("=== SAMPLE HASIL ===")
print(orders[['order_id', 'order_datetime', 'order_hour']].head(10))
print()
print("=== DISTRIBUSI JAM ===")
print(orders['order_hour'].value_counts().sort_index())

# Re-import ke SQLite (replace tabel orders saja)
conn = sqlite3.connect('db_order.db')
orders.to_sql('orders', conn, if_exists='replace', index=False)
conn.close()

print()
print("✅ Tabel orders berhasil diperbarui dengan data jam.")

=== SAMPLE HASIL ===
        order_id       order_datetime  order_hour
0   F-2616596270  2024-10-02 18:55:00          18
3   F-2622650317  2024-10-07 22:18:00          22
6   F-2625760663  2024-10-10 20:25:00          20
9   F-2626691980  2024-10-11 18:24:00          18
12  F-2629362503  2024-10-13 19:17:00          19
15  F-2631519399  2024-10-15 19:20:00          19
18  F-2632771106  2024-10-16 21:51:00          21
21  F-2633750482  2024-10-17 20:09:00          20
24  F-2633790827  2024-10-17 20:41:00          20
27  F-2634627942  2024-10-18 17:41:00          17

=== DISTRIBUSI JAM ===
order_hour
12     2
13     2
17    17
18     9
19    21
20    12
21    17
22     6
23     4
Name: count, dtype: int64

✅ Tabel orders berhasil diperbarui dengan data jam.


In [8]:
import pandas as pd
import sqlite3

# Baca dari XLSX
df = pd.read_excel('data review gofood.xlsx')

# Deduplikasi per order
orders = df.drop_duplicates(subset='Nomor Pesanan').copy()
orders = orders[[
    'Nomor Pesanan', 'Tanggal Transaksi', 'Rating untuk Resto', 'Review', 'Tag yang Dipilih'
]]
orders.columns = ['order_id', 'order_datetime', 'rating', 'review', 'tags_raw']

# Simpan jam dengan benar
orders['order_datetime'] = pd.to_datetime(orders['order_datetime']).dt.strftime('%Y-%m-%d %H:%M:%S')
orders['order_date'] = pd.to_datetime(orders['order_datetime']).dt.strftime('%Y-%m-%d')
orders['order_hour'] = pd.to_datetime(orders['order_datetime']).dt.hour
orders['order_time'] = pd.to_datetime(orders['order_datetime']).dt.strftime('%H:%M')  # tambahan jam:menit
orders['has_review'] = orders['review'].notna().astype(int)
orders['has_tags'] = orders['tags_raw'].notna().astype(int)

# Cek dulu
print(orders[['order_id', 'order_datetime', 'order_hour', 'order_time']].head(10))

        order_id       order_datetime  order_hour order_time
0   F-2616596270  2024-10-02 18:55:00          18      18:55
3   F-2622650317  2024-10-07 22:18:00          22      22:18
6   F-2625760663  2024-10-10 20:25:00          20      20:25
9   F-2626691980  2024-10-11 18:24:00          18      18:24
12  F-2629362503  2024-10-13 19:17:00          19      19:17
15  F-2631519399  2024-10-15 19:20:00          19      19:20
18  F-2632771106  2024-10-16 21:51:00          21      21:51
21  F-2633750482  2024-10-17 20:09:00          20      20:09
24  F-2633790827  2024-10-17 20:41:00          20      20:41
27  F-2634627942  2024-10-18 17:41:00          17      17:41


In [9]:
# Import ke SQLite
conn = sqlite3.connect('db_order.db')
orders.to_sql('orders', conn, if_exists='replace', index=False)
conn.close()
print("✅ Selesai")

✅ Selesai
